# 04 — Feature Engineering

## Projet : AI Business Advisor V2

### Objectif

Cette étape prépare les variables nécessaires aux futurs modèles
Machine Learning.

Trois problèmes sont traités séparément :

1. Claim occurrence :
   prédire si une police aura au moins un sinistre.

2. Claim frequency :
   modéliser le nombre de sinistres en tenant compte de l'exposition.

3. Fraud detection :
   prédire si une déclaration est frauduleuse.

Les transformations réalisées dans ce notebook sont uniquement
déterministes et basées sur des informations réellement observées.

Les transformations apprises sur les données, telles que :

- imputation ;
- standardisation ;
- One-Hot Encoding ;
- SMOTE ;
- sélection statistique de variables ;

seront effectuées après la séparation train/test afin d'éviter
le data leakage.

In [1]:
import pandas as pd
import numpy as np

import json

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

In [2]:
cwd = Path.cwd()

possible_paths = [
    cwd / "data" / "interim",
    cwd / "data-science" / "data" / "interim",
    cwd.parent / "data" / "interim"
]

INTERIM_DIR = None

for path in possible_paths:
    if path.exists():
        INTERIM_DIR = path
        break

if INTERIM_DIR is None:
    raise FileNotFoundError(
        "Impossible de localiser data/interim"
    )

DATA_DIR = INTERIM_DIR.parent

PROCESSED_DIR = DATA_DIR / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("INTERIM   :", INTERIM_DIR.resolve())
print("PROCESSED :", PROCESSED_DIR.resolve())

INTERIM   : C:\Users\user\ai-business-advisor-v2\data-science\data\interim
PROCESSED : C:\Users\user\ai-business-advisor-v2\data-science\data\processed


In [3]:
FREQ_CLEAN_PATH = (
    INTERIM_DIR
    / "freMTPL2freq_clean.csv"
)

FRAUD_CLEAN_PATH = (
    INTERIM_DIR
    / "vehicle_claim_fraud_clean.csv"
)

freq = pd.read_csv(
    FREQ_CLEAN_PATH
)

fraud = pd.read_csv(
    FRAUD_CLEAN_PATH
)

print(
    "freMTPL2 :",
    freq.shape
)

print(
    "Fraud :",
    fraud.shape
)

freMTPL2 : (676789, 12)
Fraud : (15420, 33)


In [7]:
print(
    "Colonnes freMTPL2 :"
)

print(
    freq.columns.tolist()
)

print()

print(
    "Colonnes Fraud :"
)

print(
    fraud.columns.tolist()
)
print(
    freq.dtypes
)
print(
    fraud.dtypes
)


Colonnes freMTPL2 :
['IDpol', 'ClaimNb', 'Exposure', 'Area', 'VehPower', 'VehAge', 'DrivAge', 'BonusMalus', 'VehBrand', 'VehGas', 'Density', 'Region']

Colonnes Fraud :
['Month', 'WeekOfMonth', 'DayOfWeek', 'Make', 'AccidentArea', 'DayOfWeekClaimed', 'MonthClaimed', 'WeekOfMonthClaimed', 'Sex', 'MaritalStatus', 'Age', 'Fault', 'PolicyType', 'VehicleCategory', 'VehiclePrice', 'FraudFound_P', 'PolicyNumber', 'RepNumber', 'Deductible', 'DriverRating', 'Days_Policy_Accident', 'Days_Policy_Claim', 'PastNumberOfClaims', 'AgeOfVehicle', 'AgeOfPolicyHolder', 'PoliceReportFiled', 'WitnessPresent', 'AgentType', 'NumberOfSuppliments', 'AddressChange_Claim', 'NumberOfCars', 'Year', 'BasePolicy']
IDpol           int64
ClaimNb         int64
Exposure      float64
Area              str
VehPower        int64
VehAge          int64
DrivAge         int64
BonusMalus      int64
VehBrand          str
VehGas            str
Density         int64
Region            str
dtype: object
Month                       s

# Partie A — Claim Occurrence

## Problème

Nous voulons prédire si une police observe au moins un sinistre.

La cible est construite directement à partir de `ClaimNb` :

`has_claim = 1` si `ClaimNb > 0`

`has_claim = 0` sinon.

Cette cible n'est pas artificielle : elle est dérivée d'une observation
réelle du nombre de sinistres.

Important :

`ClaimNb` ne pourra ensuite jamais être utilisé comme feature,
car il définit directement la cible.

In [ ]:
freq_fe = freq.copy()

freq_fe["has_claim"] = (
    freq_fe["ClaimNb"] > 0
).astype("int8")

freq_fe[
    [
        "ClaimNb",
        "has_claim"
    ]
].head(20)



has_claim
0    642780
1     34009
Name: count, dtype: int64

In [11]:
freq_fe[
    "has_claim"
].value_counts()

has_claim
0    642780
1     34009
Name: count, dtype: int64

In [12]:
freq_fe[
    "has_claim"
].value_counts(
    normalize=True
) * 100

has_claim
0    94.974948
1     5.025052
Name: proportion, dtype: float64

In [13]:
occurrence_excluded_features = [
    "IDpol",
    "ClaimNb"
]

In [15]:
occurrence_numeric_features = [
    "Exposure",
    "VehPower",
    "VehAge",
    "DrivAge",
    "BonusMalus",
    "Density"
]

occurrence_categorical_features = [
    "Area",
    "VehBrand",
    "VehGas",
    "Region"
]

In [17]:
freq_fe["log_density"] = np.log1p(
    freq_fe["Density"]
)

freq_fe[
    [
        "Density",
        "log_density"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
Density,676789.0,1791.924498,3959.038575,1.000000,92.000000,393.000000,1635.00000,27000.000000
log_density,676789.0,5.991589,1.856341,0.693147,4.532599,5.976351,7.40001,10.203629


In [18]:
print(
    "Skew Density :",
    freq_fe["Density"].skew()
)

print(
    "Skew log_density :",
    freq_fe["log_density"].skew()
)

Skew Density : 4.651821324429923
Skew log_density : 0.050734096963882806


In [19]:
occurrence_numeric_features = [
    "Exposure",
    "VehPower",
    "VehAge",
    "DrivAge",
    "BonusMalus",
    "log_density"
]

occurrence_categorical_features = [
    "Area",
    "VehBrand",
    "VehGas",
    "Region"
]

occurrence_features = (
    occurrence_numeric_features
    +
    occurrence_categorical_features
)

In [21]:
X_occurrence = (
    freq_fe[
        occurrence_features
    ].copy()
)

y_occurrence = (
    freq_fe[
        "has_claim"
    ].copy()
)

print(
    "X occurrence :",
    X_occurrence.shape
)

print(
    "y occurrence :",
    y_occurrence.shape
)

X occurrence : (676789, 10)
y occurrence : (676789,)


In [22]:
assert "ClaimNb" not in X_occurrence.columns
assert "has_claim" not in X_occurrence.columns
assert "IDpol" not in X_occurrence.columns

print(
    "Contrôle leakage occurrence : OK"
)

Contrôle leakage occurrence : OK


# Partie B — Claim Frequency

Le deuxième problème est différent.

Nous ne voulons plus répondre uniquement :

« Le client a-t-il eu un sinistre ? »

Nous voulons modéliser :

« Combien de sinistres sont attendus compte tenu de la durée d'exposition ? »

La variable cible est :

`ClaimNb`

L'exposition est :

`Exposure`

L'exposition ne doit pas être traitée exactement comme une feature
classique. Elle représentera la durée pendant laquelle la police
est exposée au risque.

In [23]:
freq_fe["claim_frequency"] = (
    freq_fe["ClaimNb"]
    /
    freq_fe["Exposure"]
)

In [27]:
y_frequency = (
    freq_fe[
        "ClaimNb"
    ].copy()
)

exposure_frequency = (
    freq_fe[
        "Exposure"
    ].copy()
)

frequency_numeric_features = [
    "VehPower",
    "VehAge",
    "DrivAge",
    "BonusMalus",
    "log_density"
]

frequency_categorical_features = [
    "Area",
    "VehBrand",
    "VehGas",
    "Region"
]

frequency_features = (
    frequency_numeric_features
    +
    frequency_categorical_features
)

X_frequency = (
    freq_fe[
        frequency_features
    ].copy()
)

In [28]:
assert "ClaimNb" not in X_frequency.columns
assert "Exposure" not in X_frequency.columns
assert "IDpol" not in X_frequency.columns

print(
    "Contrôle leakage frequency : OK"
)

Contrôle leakage frequency : OK


In [29]:
y_frequency.describe()

count    676789.000000
mean          0.053263
std           0.240157
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          16.000000
Name: ClaimNb, dtype: float64

In [30]:
y_frequency.value_counts(
    normalize=True
).sort_index() * 100

ClaimNb
0     94.974948
1      4.747418
2      0.263154
3      0.012116
4      0.001034
5      0.000296
6      0.000148
8      0.000148
9      0.000148
11     0.000443
16     0.000148
Name: proportion, dtype: float64

# Partie C — Fraud Detection

La cible est :

`FraudFound_P`

Le problème est une classification binaire fortement déséquilibrée.

L'objectif du feature engineering est maintenant de :

- identifier les identifiants ;
- identifier les variables numériques ;
- identifier les variables catégorielles ;
- identifier les variables ordinales ;
- détecter les variables redondantes ;
- identifier les variables potentiellement sujettes au data leakage ;
- préparer un feature set conservateur.

In [32]:
fraud_fe = fraud.copy()
fraud_fe[
    "FraudFound_P"
].value_counts()

FraudFound_P
0    14497
1      923
Name: count, dtype: int64

In [33]:
fraud_fe[
    "FraudFound_P"
].value_counts(
    normalize=True
) * 100

FraudFound_P
0    94.014267
1     5.985733
Name: proportion, dtype: float64

In [36]:
fraud_target = (
    "FraudFound_P"
)

fraud_id_features = [
    "PolicyNumber"
]

fraud_review_features = [
    "RepNumber"
]

In [37]:
potential_leakage_features = [
    "Fault",
    "PoliceReportFiled",
    "WitnessPresent",
    "NumberOfSuppliments",
    "Days_Policy_Claim",
    "AddressChange_Claim"
]

In [38]:
policy_relation = (
    fraud_fe
    .groupby(
        "PolicyType"
    )
    .agg(
        vehicle_categories=(
            "VehicleCategory",
            "nunique"
        ),
        base_policies=(
            "BasePolicy",
            "nunique"
        )
    )
)

policy_relation

,vehicle_categories,base_policies
PolicyType,,
Sedan - All Perils,1,1
Sedan - Collision,1,1
Sedan - Liability,1,1
Sport - All Perils,1,1
Sport - Collision,1,1
Sport - Liability,1,1
Utility - All Perils,1,1
Utility - Collision,1,1
Utility - Liability,1,1


In [39]:
reverse_relation = (
    fraud_fe
    .groupby(
        [
            "VehicleCategory",
            "BasePolicy"
        ]
    )[
        "PolicyType"
    ]
    .nunique()
)

reverse_relation.sort_values(
    ascending=False
).head()

VehicleCategory  BasePolicy
Sport            Liability     2
Sedan            All Perils    1
Sport            All Perils    1
Sedan            Collision     1
Sport            Collision     1
Name: PolicyType, dtype: int64

In [41]:
fraud_redundant_features = [
    "PolicyType"
]

fraud_split_features = [
    "Year"
]

In [44]:
fraud_excluded_features = (
    [
        fraud_target
    ]
    +
    fraud_id_features
    +
    fraud_review_features
    +
    potential_leakage_features
    +
    fraud_redundant_features
    +
    fraud_split_features
)

fraud_candidate_features = [
    col
    for col in fraud_fe.columns
    if col not in fraud_excluded_features
]

print(
    "Nombre de features candidates :",
    len(fraud_candidate_features)
)

print()

print(
    fraud_candidate_features
)

Nombre de features candidates : 22

['Month', 'WeekOfMonth', 'DayOfWeek', 'Make', 'AccidentArea', 'DayOfWeekClaimed', 'MonthClaimed', 'WeekOfMonthClaimed', 'Sex', 'MaritalStatus', 'Age', 'VehicleCategory', 'VehiclePrice', 'Deductible', 'DriverRating', 'Days_Policy_Accident', 'PastNumberOfClaims', 'AgeOfVehicle', 'AgeOfPolicyHolder', 'AgentType', 'NumberOfCars', 'BasePolicy']


In [45]:
fraud_fe[
    fraud_candidate_features
].dtypes

Month                       str
WeekOfMonth               int64
DayOfWeek                   str
Make                        str
AccidentArea                str
DayOfWeekClaimed            str
MonthClaimed                str
WeekOfMonthClaimed        int64
Sex                         str
MaritalStatus               str
Age                     float64
VehicleCategory             str
VehiclePrice                str
Deductible                int64
DriverRating              int64
Days_Policy_Accident        str
PastNumberOfClaims          str
AgeOfVehicle                str
AgeOfPolicyHolder           str
AgentType                   str
NumberOfCars                str
BasePolicy                  str
dtype: object

In [49]:
fraud_numeric_features = [
    "Age",
    "WeekOfMonth",
    "WeekOfMonthClaimed",
    "Deductible",
    "DriverRating"
]

fraud_continuous_features = [
    "Age",
    "Deductible"
]

fraud_ordinal_features = [
    "WeekOfMonth",
    "WeekOfMonthClaimed",
    "DriverRating"
]

fraud_categorical_features = [
    col
    for col in fraud_candidate_features
    if col not in (
        fraud_continuous_features
        +
        fraud_ordinal_features
    )
]

print(
    "Continuous :",
    fraud_continuous_features
)

print()

print(
    "Ordinal :",
    fraud_ordinal_features
)

print()

print(
    "Categorical :",
    fraud_categorical_features
)

Continuous : ['Age', 'Deductible']

Ordinal : ['WeekOfMonth', 'WeekOfMonthClaimed', 'DriverRating']

Categorical : ['Month', 'DayOfWeek', 'Make', 'AccidentArea', 'DayOfWeekClaimed', 'MonthClaimed', 'Sex', 'MaritalStatus', 'VehicleCategory', 'VehiclePrice', 'Days_Policy_Accident', 'PastNumberOfClaims', 'AgeOfVehicle', 'AgeOfPolicyHolder', 'AgentType', 'NumberOfCars', 'BasePolicy']


In [50]:
fraud_fe["Age"].isna().sum()

np.int64(320)

In [52]:
X_fraud = (
    fraud_fe[
        fraud_candidate_features
    ].copy()
)

y_fraud = (
    fraud_fe[
        fraud_target
    ].copy()
)

print(
    "X fraud :",
    X_fraud.shape
)

print(
    "y fraud :",
    y_fraud.shape
)

X fraud : (15420, 22)
y fraud : (15420,)


In [53]:
assert "FraudFound_P" not in X_fraud.columns
assert "PolicyNumber" not in X_fraud.columns
assert "Year" not in X_fraud.columns

for feature in potential_leakage_features:

    assert feature not in X_fraud.columns

print(
    "Contrôle leakage Fraud baseline : OK"
)

Contrôle leakage Fraud baseline : OK


In [54]:
fraud_cardinality = (
    X_fraud[
        fraud_categorical_features
    ]
    .nunique()
    .sort_values(
        ascending=False
    )
)

fraud_cardinality

Make                    19
Month                   12
MonthClaimed            12
AgeOfPolicyHolder        9
AgeOfVehicle             8
DayOfWeekClaimed         7
DayOfWeek                7
VehiclePrice             6
Days_Policy_Accident     5
NumberOfCars             5
PastNumberOfClaims       4
MaritalStatus            4
VehicleCategory          3
BasePolicy               3
AccidentArea             2
Sex                      2
AgentType                2
dtype: int64

In [55]:
freq_cardinality = (
    X_occurrence[
        occurrence_categorical_features
    ]
    .nunique()
)

freq_cardinality

Area         6
VehBrand    11
VehGas       2
Region      22
dtype: int64

In [59]:
feature_manifest = {

    "claim_occurrence": {

        "target":
            "has_claim",

        "numeric_features":
            occurrence_numeric_features,

        "categorical_features":
            occurrence_categorical_features,

        "excluded_features": [
            "IDpol",
            "ClaimNb"
        ]
    },

    "claim_frequency": {

        "target":
            "ClaimNb",

        "exposure":
            "Exposure",

        "numeric_features":
            frequency_numeric_features,

        "categorical_features":
            frequency_categorical_features,

        "excluded_features": [
            "IDpol"
        ]
    },

    "fraud_detection": {

        "target":
            fraud_target,

        "continuous_features":
            fraud_continuous_features,

        "ordinal_features":
            fraud_ordinal_features,

        "categorical_features":
            fraud_categorical_features,

        "id_features":
            fraud_id_features,

        "potential_leakage_features":
            potential_leakage_features,

        "redundant_features":
            fraud_redundant_features,

        "split_features":
            fraud_split_features
    }
}

METADATA_DIR = (
    DATA_DIR
    / "metadata"
)

METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MANIFEST_PATH = (
    METADATA_DIR
    / "feature_manifest.json"
)

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        feature_manifest,
        f,
        indent=4,
        ensure_ascii=False
    )

print(
    MANIFEST_PATH
)

print(
    json.dumps(
        feature_manifest,
        indent=4,
        ensure_ascii=False
    )
)

c:\Users\user\ai-business-advisor-v2\data-science\data\metadata\feature_manifest.json
{
    "claim_occurrence": {
        "target": "has_claim",
        "numeric_features": [
            "Exposure",
            "VehPower",
            "VehAge",
            "DrivAge",
            "BonusMalus",
            "log_density"
        ],
        "categorical_features": [
            "Area",
            "VehBrand",
            "VehGas",
            "Region"
        ],
        "excluded_features": [
            "IDpol",
            "ClaimNb"
        ]
    },
    "claim_frequency": {
        "target": "ClaimNb",
        "exposure": "Exposure",
        "numeric_features": [
            "VehPower",
            "VehAge",
            "DrivAge",
            "BonusMalus",
            "log_density"
        ],
        "categorical_features": [
            "Area",
            "VehBrand",
            "VehGas",
            "Region"
        ],
        "excluded_features": [
            "IDpol"
        ]
    }

In [61]:
occurrence_modeling_base = (
    freq_fe[
        [
            "IDpol"
        ]
        +
        occurrence_features
        +
        [
            "has_claim"
        ]
    ]
    .copy()
)

occurrence_modeling_base.head()

,IDpol,Exposure,VehPower,VehAge,DrivAge,BonusMalus,log_density,Area,VehBrand,VehGas,Region,has_claim
0,1,0.10,5,0,55,50,7.104965,D,B12,Regular,R82,1
1,3,0.77,5,0,55,50,7.104965,D,B12,Regular,R82,1
2,5,0.75,6,2,52,50,4.007333,B,B12,Diesel,R22,1
3,10,0.09,7,0,46,50,4.343805,B,B12,Diesel,R72,1
4,11,0.84,7,0,46,50,4.343805,B,B12,Diesel,R72,1


In [63]:
frequency_modeling_base = (
    freq_fe[
        [
            "IDpol",
            "Exposure"
        ]
        +
        frequency_features
        +
        [
            "ClaimNb"
        ]
    ]
    .copy()
)
frequency_modeling_base.head()

,IDpol,Exposure,VehPower,VehAge,DrivAge,BonusMalus,log_density,Area,VehBrand,VehGas,Region,ClaimNb
0,1,0.10,5,0,55,50,7.104965,D,B12,Regular,R82,1
1,3,0.77,5,0,55,50,7.104965,D,B12,Regular,R82,1
2,5,0.75,6,2,52,50,4.007333,B,B12,Diesel,R22,1
3,10,0.09,7,0,46,50,4.343805,B,B12,Diesel,R72,1
4,11,0.84,7,0,46,50,4.343805,B,B12,Diesel,R72,1


In [64]:
fraud_modeling_base = (
    fraud_fe[
        [
            "PolicyNumber",
            "Year"
        ]
        +
        fraud_candidate_features
        +
        [
            "FraudFound_P"
        ]
    ]
    .copy()
)

In [65]:
assert (
    not occurrence_modeling_base
    .columns
    .duplicated()
    .any()
)

assert (
    not frequency_modeling_base
    .columns
    .duplicated()
    .any()
)

assert (
    not fraud_modeling_base
    .columns
    .duplicated()
    .any()
)

print(
    "Colonnes dupliquées : aucune"
)

Colonnes dupliquées : aucune


In [66]:
print(
    occurrence_modeling_base.shape
)

print()

print(
    occurrence_modeling_base
    .isna()
    .sum()
)

(676789, 12)

IDpol          0
Exposure       0
VehPower       0
VehAge         0
DrivAge        0
BonusMalus     0
log_density    0
Area           0
VehBrand       0
VehGas         0
Region         0
has_claim      0
dtype: int64


In [67]:
print(
    frequency_modeling_base.shape
)

print()

print(
    frequency_modeling_base[
        [
            "Exposure",
            "ClaimNb"
        ]
    ].describe()
)

(676789, 12)

            Exposure        ClaimNb
count  676789.000000  676789.000000
mean        0.527692       0.053263
std         0.363857       0.240157
min         0.002732       0.000000
25%         0.170000       0.000000
50%         0.490000       0.000000
75%         0.990000       0.000000
max         1.000000      16.000000


In [68]:
print(
    fraud_modeling_base.shape
)

print()

print(
    fraud_modeling_base
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
    .head(10)
)

(15420, 25)

Age                 320
MonthClaimed          1
DayOfWeekClaimed      1
Year                  0
PolicyNumber          0
DayOfWeek             0
WeekOfMonth           0
AccidentArea          0
Month                 0
Make                  0
dtype: int64


In [69]:
OCCURRENCE_PATH = (
    PROCESSED_DIR
    / "claim_occurrence_modeling_base.csv"
)

FREQUENCY_PATH = (
    PROCESSED_DIR
    / "claim_frequency_modeling_base.csv"
)

FRAUD_PATH = (
    PROCESSED_DIR
    / "fraud_modeling_base.csv"
)

In [70]:
occurrence_modeling_base.to_csv(
    OCCURRENCE_PATH,
    index=False
)

frequency_modeling_base.to_csv(
    FREQUENCY_PATH,
    index=False
)

fraud_modeling_base.to_csv(
    FRAUD_PATH,
    index=False
)

In [71]:
occurrence_check = pd.read_csv(
    OCCURRENCE_PATH
)

frequency_check = pd.read_csv(
    FREQUENCY_PATH
)

fraud_check = pd.read_csv(
    FRAUD_PATH
)

print(
    occurrence_check.shape
)

print(
    frequency_check.shape
)

print(
    fraud_check.shape
)

(676789, 12)
(676789, 12)
(15420, 25)


# Data Leakage Controls

## Claim Occurrence

`ClaimNb` est exclu des features car il sert directement à construire
la cible `has_claim`.

## Claim Frequency

`ClaimNb` est la cible.

Toute variable calculée à partir de `ClaimNb`, telle que
`claim_frequency = ClaimNb / Exposure`, ne peut pas être utilisée
comme variable explicative.

`Exposure` est conservée séparément afin d'être intégrée correctement
dans le modèle de fréquence.

## Fraud Detection

`FraudFound_P` est la cible et n'apparaît jamais dans X.

`PolicyNumber` est exclu comme identifiant.

Les variables potentiellement indisponibles au moment de la prédiction
sont exclues de la première baseline et devront être validées avant
toute intégration.

# Conclusion — Feature Engineering

Trois problèmes Machine Learning indépendants sont maintenant définis.

## Claim Occurrence

Target :

`has_claim`

Features :

- Exposure
- VehPower
- VehAge
- DrivAge
- BonusMalus
- log_density
- Area
- VehBrand
- VehGas
- Region

`ClaimNb` et `IDpol` sont exclus des features.

---

## Claim Frequency

Target :

`ClaimNb`

Exposure :

`Exposure`

Features :

- VehPower
- VehAge
- DrivAge
- BonusMalus
- log_density
- Area
- VehBrand
- VehGas
- Region

L'exposition sera intégrée explicitement dans le futur modèle.

---

## Fraud Detection

Target :

`FraudFound_P`

Les identifiants ont été séparés des features.

Les variables ont été classifiées en :

- continues ;
- ordinales ;
- catégorielles.

Les variables potentiellement sujettes au data leakage ont été
exclues de la première baseline.

`Year` est conservé pour étudier un protocole d'évaluation temporel.

---

Aucune transformation apprise sur les données n'a été appliquée.

L'étape suivante consiste à définir le protocole expérimental :

`05_experimental_protocol.ipynb`

Nous définirons notamment :

- train / validation / test ;
- split aléatoire ou temporel ;
- stratification ;
- cross-validation ;
- métriques ;
- preprocessing pipelines ;
- baselines ;
- règles pour éviter le data leakage.